# Assignment 2B — RAG Pipeline
**Name:** TODO  |  **Domain:** TODO

Pipeline: chunk → dense search (FAISS) → BM25 + hybrid (RRF) → cross-encoder rerank → grounded generation → LLM judge.

Cells marked `TODO` need your input. Run top to bottom so all outputs are visible.

## 0. Setup

In [ ]:
!pip install -q sentence-transformers faiss-cpu rank_bm25 transformers accelerate pandas

In [ ]:
import re, time, glob, os, gc
import numpy as np, pandas as pd, faiss, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

# ---------- TODO: configuration ----------
CORPUS_DIR   = "corpus"                 # folder with your cleaned Assignment 1 .txt files
EMBED_MODEL  = "BAAI/bge-small-en-v1.5" # max input 512 tokens (MiniLM-L6 is only 256)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # bge query prefix; use "" for other models
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GEN_MODEL    = "path-or-hub-id-of-your-assignment-1-model"   # TODO
JUDGE_MODEL  = "Qwen/Qwen2.5-7B-Instruct"                    # must differ from generator
BATCH_SIZE   = 64
# -----------------------------------------

## Part A — Dense search and chunk size (5 marks)
### Answer spans
For each of your 10 queries, paste the **exact** sentence/passage from your corpus that answers it (copy-paste from the .txt file). A chunk is a hit if it contains this span (whitespace and case are ignored).

In [ ]:
# TODO: replace with your 10 queries from the Assignment 1 evaluation set
EVAL = [
    {"query": "TODO question 1", "expected": "TODO expected answer 1", "span": "TODO exact answer span from corpus 1"},
    # ... up to 10 entries
]
# One query whose answer is NOT in the corpus (used in D1.3)
NOT_IN_CORPUS = "TODO a question your corpus cannot answer"

docs = {}
for p in sorted(glob.glob(f"{CORPUS_DIR}/*.txt")):
    docs[os.path.basename(p)] = open(p, encoding="utf-8").read()
print(len(docs), "documents,", sum(len(t) for t in docs.values()), "characters")

norm = lambda s: re.sub(r"\s+", " ", s).lower().strip()
all_text = norm(" ".join(docs.values()))
for i, e in enumerate(EVAL, 1):
    print(f"Q{i} span found in corpus:", norm(e["span"]) in all_text)   # every line must say True

In [ ]:
embedder = SentenceTransformer(EMBED_MODEL)
tok = embedder.tokenizer
MAX_LEN = embedder.max_seq_length
print("Embedding model:", EMBED_MODEL, "| max input length:", MAX_LEN, "tokens")

def count_tokens(text):
    return len(tok(text, add_special_tokens=False)["input_ids"])

def is_hit(chunk_text, span):
    return norm(span) in norm(chunk_text)

### A1. Dense search (2 marks)

In [ ]:
def chunk_corpus(docs, size):
    """Fixed-size token chunks, no overlap. Uses character offsets so chunk text is exact."""
    chunks = []
    for name, text in docs.items():
        offs = tok(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
        for i in range(0, len(offs), size):
            seg = offs[i:i + size]
            chunks.append({"doc": name, "text": text[seg[0][0]:seg[-1][1]], "n_tokens": len(seg)})
    return chunks

class DenseIndex:
    """Embeds chunks, builds FAISS IndexFlatIP, returns top-k chunks for a query."""
    def __init__(self, chunks, model=embedder, batch_size=BATCH_SIZE):
        self.chunks, self.model = chunks, model
        t0 = time.time()
        emb = model.encode([c["text"] for c in chunks], batch_size=batch_size,
                           normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        self.dim = emb.shape[1]
        self.index = faiss.IndexFlatIP(self.dim)   # inner product on normalised vectors = cosine
        self.index.add(emb)
        self.build_time = time.time() - t0
    def search(self, query, k=5):
        q = self.model.encode([QUERY_PREFIX + query], normalize_embeddings=True).astype("float32")
        scores, ids = self.index.search(q, k)
        return [int(i) for i in ids[0] if i >= 0]

### A2. Choose a chunk size (3 marks)
Both sizes must be ≤ the embedding model's limit, otherwise text is silently truncated.

In [ ]:
def hit_rate(results, k):
    # results: list of lists of chunk texts (one list per query, in rank order)
    return 100 * np.mean([any(is_hit(t, e["span"]) for t in r[:k]) for r, e in zip(results, EVAL)])

rows, indexes = [], {}
for size in [100, 400]:
    assert size <= MAX_LEN - 2, f"{size} tokens exceeds the model limit ({MAX_LEN}); switch model or change size"
    chunks = chunk_corpus(docs, size)
    idx = DenseIndex(chunks); indexes[size] = idx
    res = [[chunks[i]["text"] for i in idx.search(e["query"], 3)] for e in EVAL]
    tokens_per_q = np.mean([sum(count_tokens(t) for t in r) for r in res])
    rows.append({"Chunk size (tokens)": size, "Total chunks": len(chunks),
                 "Hit rate@3 (%)": hit_rate(res, 3), "Tokens sent per query (top 3)": round(tokens_per_q, 1)})
pd.DataFrame(rows)

In [ ]:
# A1.2 report (for the FINAL chunk set — set CHUNK_SIZE after reading the table above)
CHUNK_SIZE = 400   # TODO: your choice
final = indexes[CHUNK_SIZE]; chunks = final.chunks
print("Embedding model :", EMBED_MODEL)
print("Vector size     :", final.dim)
print("Similarity      : cosine (L2-normalised vectors + FAISS IndexFlatIP inner product)")
print("Batch size      :", BATCH_SIZE)
print(f"Index build time: {final.build_time:.2f} s for {len(chunks)} chunks")

**A2.2 — Chosen chunk size:** TODO (2–3 sentences: compare hit rate@3, tokens per query, and how much context your Assignment 1 model can handle. If you changed a size or model because of the embedding limit, say so here.)

## Part B — Keyword and hybrid search (4 marks)

In [ ]:
dense = final
def bm25_tokens(s): return re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([bm25_tokens(c["text"]) for c in chunks])

def dense_ids(q, k=5): return dense.search(q, k)
def bm25_ids(q, k=5):
    s = bm25.get_scores(bm25_tokens(q)); return [int(i) for i in np.argsort(-s)[:k]]

def rrf_fuse(lists, k_const=60, top=5):
    scores = {}
    for lst in lists:
        for rank, cid in enumerate(lst, 1):          # ranks start at 1; absent chunk adds nothing
            scores[cid] = scores.get(cid, 0) + 1 / (k_const + rank)
    return [c for c, _ in sorted(scores.items(), key=lambda x: -x[1])[:top]]

def hybrid_ids(q, k=5):
    return rrf_fuse([dense_ids(q, 20), bm25_ids(q, 20)], top=k)   # fuse top 20 of each

def evaluate(fn, k=5):
    fn(EVAL[0]["query"], k)                           # warm-up run, not timed
    lat, res = [], []
    for e in EVAL:
        t0 = time.perf_counter(); ids = fn(e["query"], k); lat.append((time.perf_counter() - t0) * 1000)
        res.append([chunks[i]["text"] for i in ids])
    return {"Average query latency (ms)": round(np.mean(lat), 2),
            "Hit rate@1 (%)": hit_rate(res, 1), "Hit rate@3 (%)": hit_rate(res, 3)}, res

methods = {"Dense": dense_ids, "BM25": bm25_ids, "Hybrid (RRF)": hybrid_ids}
table, results_B = {}, {}
for name, fn in methods.items():
    table[name], results_B[name] = evaluate(fn)
pd.DataFrame(table).T

**B4 — Best method:** TODO (2–3 sentences. Name the winner and explain with ONE query, e.g. "Query 4 contains the exact term X, which BM25 matched but dense ranked 3rd…". Print per-query hits below to find a good example.)

In [ ]:
# helper for choosing an example query: which method hit at rank 1 for each query?
for i, e in enumerate(EVAL, 1):
    print(i, {m: is_hit(results_B[m][i-1][0], e["span"]) for m in methods})

## Part C — Reranking (4 marks)

In [ ]:
BEST = "Hybrid (RRF)"   # TODO: set to the winner of Part B
best_fn = methods[BEST]
reranker = CrossEncoder(RERANK_MODEL)

def rerank(query, k_in=10, k_out=3):
    ids = best_fn(query, k_in)
    scores = reranker.predict([(query, chunks[i]["text"]) for i in ids])
    return ids, [ids[j] for j in np.argsort(-scores)[:k_out]]

rerank(EVAL[0]["query"])   # warm-up
before, after, times = [], [], []
for e in EVAL:
    t0 = time.perf_counter()
    ids = best_fn(e["query"], 10)
    t1 = time.perf_counter()
    sc = reranker.predict([(e["query"], chunks[i]["text"]) for i in ids])
    times.append((time.perf_counter() - t1) * 1000)       # reranking time only
    before.append(ids[:3]); after.append([ids[j] for j in np.argsort(-sc)[:3]])

for i, e in enumerate(EVAL, 1):
    print(f"Q{i}: {e['query']}\n   before: {before[i-1]}  after: {after[i-1]}")

In [ ]:
h1 = lambda lists: 100 * np.mean([is_hit(chunks[l[0]]["text"], e["span"]) for l, e in zip(lists, EVAL)])
change = 100 * np.mean([b[0] != a[0] for b, a in zip(before, after)])
pd.DataFrame({"Measure": ["Reranking time per query (ms)", "Rank change rate (%)",
                          "Hit rate@1 before reranking (from Part B)", "Hit rate@1 after reranking"],
              "Value": [round(np.mean(times), 1), change, h1(before), h1(after)]})

**C4 — Did reranking help?** TODO (2–3 sentences: hit rate@1 before vs after, and the ms it added per query.)

## Part D — Generate and check answers (7 marks)

### D1. Grounded answers

In [ ]:
SYSTEM_PROMPT = """You are a careful assistant. Answer the question using ONLY the numbered chunks below.
Rules:
1. Do not use any outside knowledge.
2. Cite the chunk number(s) you used in square brackets, e.g. [2].
3. If the chunks do not contain the answer, reply exactly: Not found in the documents."""

def build_messages(query, chunk_ids):
    ctx = "\n\n".join(f"[{n}] {chunks[i]['text']}" for n, i in enumerate(chunk_ids, 1))
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Chunks:\n{ctx}\n\nQuestion: {query}\nAnswer:"}]

print(SYSTEM_PROMPT)   # D1.1: show the prompt

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_llm(name):
    t = AutoTokenizer.from_pretrained(name)
    m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, device_map="auto")
    return t, m

def chat(t, m, messages, max_new_tokens=300):
    ids = t.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)   # greedy = temperature 0
    return t.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

gen_tok, gen_model = load_llm(GEN_MODEL)
print("Generator:", GEN_MODEL)

In [ ]:
answers = []
for i, (e, ids) in enumerate(zip(EVAL, after), 1):
    a = chat(gen_tok, gen_model, build_messages(e["query"], ids)); answers.append(a)
    print(f"Q{i}: {e['query']}\nA{i}: {a}\n")

In [ ]:
# D1.3 — query not in the corpus
ids_nf, top_nf = rerank(NOT_IN_CORPUS)
print("Q:", NOT_IN_CORPUS); print("A:", chat(gen_tok, gen_model, build_messages(NOT_IN_CORPUS, top_nf)))

**Only if you switched generator:** the Assignment 1 model ignored the chunks / never said "Not found" — example below, then set `GEN_MODEL` to the instruct model you used (name it here): TODO

### D2. LLM judge
Judge model: **TODO (name it; must differ from generator)**

In [ ]:
del gen_model; gc.collect(); torch.cuda.empty_cache()
judge_tok, judge_model = load_llm(JUDGE_MODEL)

JUDGE_PROMPT = """You are a strict examiner. Evaluate a RAG system's answer on two separate checks.

QUESTION: {question}
EXPECTED ANSWER: {expected}
RETRIEVED CHUNKS:
{chunks}
MODEL ANSWER: {answer}

CHECK 1 - CORRECT. Compare the model answer with the expected answer only.
- correct: contains all key facts of the expected answer and nothing contradicting it.
- partly correct: contains some but not all key facts, or is correct but missing an important detail.
- wrong: contradicts the expected answer, misses its key facts entirely, or is irrelevant.
- If the model answer is "Not found in the documents", mark wrong (the answer exists in the corpus).

CHECK 2 - SUPPORTED. Is EVERY statement in the model answer stated in the retrieved chunks above?
- yes: every statement can be found in the chunks.
- no: at least one statement is not in the chunks (even if true in reality).
- If the answer is "Not found in the documents", mark yes (it makes no unsupported claim).

Reply in EXACTLY this format, one item per line:
CORRECT: <correct|partly correct|wrong>
SUPPORTED: <yes|no>
REASON: <one sentence>
UNSUPPORTED_QUOTE: <quote of any unsupported statement, or none>"""

def judge(e, ids, answer):
    ctx = "\n".join(f"[{n}] {chunks[i]['text']}" for n, i in enumerate(ids, 1))
    p = JUDGE_PROMPT.format(question=e["query"], expected=e["expected"], chunks=ctx, answer=answer)
    raw = chat(judge_tok, judge_model, [{"role": "user", "content": p}], max_new_tokens=200)
    g = lambda key: (re.search(rf"{key}:\s*(.+)", raw) or [None, "PARSE_ERROR"])[1].strip()
    return {"correct": g("CORRECT").lower(), "supported": g("SUPPORTED").lower(),
            "reason": g("REASON"), "unsupported_quote": g("UNSUPPORTED_QUOTE"), "raw": raw}

print(JUDGE_PROMPT)

In [ ]:
verdicts = [judge(e, ids, a) for e, ids, a in zip(EVAL, after, answers)]
jt = pd.DataFrame([{"Query": e["query"], "Correct": v["correct"], "Supported": v["supported"], "Reason": v["reason"]}
                   for e, v in zip(EVAL, verdicts)])
jt.index = range(1, len(jt) + 1)
pd.set_option("display.max_colwidth", 120); display(jt)

c = jt["Correct"].value_counts()
pd.DataFrame({"Check": ["Correct", "Partly correct", "Wrong", "Supported by the chunks"],
              "Count out of 10": [c.get("correct", 0), c.get("partly correct", 0), c.get("wrong", 0),
                                  (jt["Supported"] == "yes").sum()]})

**D2.2 Spot-check (3 answers).** Read each answer and its chunks yourself.

| Query # | Judge said | I agree? | Note |
|---|---|---|---|
| TODO | | | |
| TODO | | | |
| TODO | | | |

Agreement: TODO / 3. Disagreement explained: TODO

In [ ]:
# D2.3 — show the 3 chunks of a wrong / unsupported answer
BAD = 0   # TODO: index (0-based) of a wrong or unsupported answer
print("Query   :", EVAL[BAD]["query"]); print("Expected:", EVAL[BAD]["expected"])
print("Answer  :", answers[BAD]); print("Verdict :", verdicts[BAD]["correct"], "/", verdicts[BAD]["supported"])
for n, i in enumerate(after[BAD], 1):
    print(f"\n[{n}] contains answer span? {is_hit(chunks[i]['text'], EVAL[BAD]['span'])}\n{chunks[i]['text']}")

**D2.3 Cause:** TODO — was the needed information in the 3 chunks (the "contains answer span?" line above)? If not → retrieval problem. If yes but the answer is wrong/unsupported → generation problem.

**D2.4 Suggested fix:** TODO — one change (chunk size, search method, reranking, prompt wording) and why it would fix this case.